# Building the Attention Block: Multi-Head Attention

Implement a custom multi-head attention `nn.Module`, verify it with
shape-assertion checkpoints, then confirm it's numerically correct by
comparing its output against a real Hugging Face attention layer using
shared weights. No GPU required — everything runs on small, synthetic
tensors on CPU.

## Setup

In [ ]:
!pip install torch transformers

In [ ]:
import torch
import torch.nn as nn
import math
from transformers import BertConfig
from transformers.models.bert.modeling_bert import BertSelfAttention

torch.manual_seed(0)
print("PyTorch:", torch.__version__)

## Step 2: Implement the Multi-Head Attention Class

Implement `__init__` and `forward` below. `d_model` is the total
embedding dimension; `num_heads` is how many heads to split it into.
Each head operates on `d_k = d_model // num_heads` dimensions.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # TODO: define four linear layers: W_q, W_k, W_v (each
        # d_model -> d_model), and W_o (the final output projection,
        # also d_model -> d_model)


    def forward(self, x, causal=False):
        batch_size, seq_len, _ = x.shape

        # TODO: project x through W_q, W_k, W_v, then reshape each
        # from [Batch, Seq_Len, d_model] into [Batch, Heads, Seq_Len, Dim]
        # using .view(batch_size, seq_len, self.num_heads, self.d_k)
        # followed by .transpose(1, 2)
        Q = None
        K = None
        V = None

        # TODO: compute scaled dot-product attention scores:
        # torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = None

        # TODO: if causal is True, build a lower-triangular mask with
        # torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        # and apply it to scores with masked_fill, filling masked
        # positions with float('-inf')


        # TODO: apply softmax over the last dimension, then multiply
        # by V to get the per-head context
        attn_weights = None
        context = None

        # TODO: recombine heads — transpose back to
        # [Batch, Seq_Len, Heads, Dim], call .contiguous(), then
        # .view(batch_size, seq_len, self.d_model) to merge heads
        # back into a single d_model-sized vector per position
        context = None

        # TODO: apply the final output projection W_o and return
        output = None
        return output

### Shape-assertion checkpoint — do not modify

In [ ]:
d_model, num_heads = 32, 4
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
x = torch.randn(2, 6, d_model)

# --- Test bidirectional (causal=False) ---
out_bidir = mha(x, causal=False)
assert out_bidir.shape == torch.Size([2, 6, d_model]), f"FAIL: got {out_bidir.shape}"
print("PASS: bidirectional output shape:", out_bidir.shape)

# --- Test causal (causal=True) ---
out_causal = mha(x, causal=True)
assert out_causal.shape == torch.Size([2, 6, d_model]), f"FAIL: got {out_causal.shape}"
print("PASS: causal output shape:", out_causal.shape)

print("\nBoth configurations produce the correct [Batch, Seq_Len, d_model] output shape.")

## Step 4: Verify Against a Real Hugging Face Attention Layer

This builds a real Hugging Face `BertSelfAttention` layer, copies its
weights directly into a fresh instance of your `MultiHeadAttention`
class, and compares both modules' outputs on the same input. If your
implementation is correct, the two outputs should match to within
floating-point precision.

In [ ]:
config = BertConfig(hidden_size=d_model, num_attention_heads=num_heads, num_hidden_layers=1)
hf_attn = BertSelfAttention(config)
hf_attn.eval()

verify_mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
verify_mha.eval()

# Copy Hugging Face's real weights into your custom module, so both
# are computing the same math on the same weights -- a same-weights
# comparison is the only way shapes matching also means the math matches.
with torch.no_grad():
    verify_mha.W_q.weight.copy_(hf_attn.query.weight)
    verify_mha.W_q.bias.copy_(hf_attn.query.bias)
    verify_mha.W_k.weight.copy_(hf_attn.key.weight)
    verify_mha.W_k.bias.copy_(hf_attn.key.bias)
    verify_mha.W_v.weight.copy_(hf_attn.value.weight)
    verify_mha.W_v.bias.copy_(hf_attn.value.bias)
    # Note: BertSelfAttention has no output projection of its own --
    # it returns the raw per-head context, recombined. To compare
    # apples to apples, zero out W_o so it acts as an identity pass-through.
    verify_mha.W_o.weight.copy_(torch.eye(d_model))
    verify_mha.W_o.bias.zero_()

x_verify = torch.randn(2, 6, d_model)

with torch.no_grad():
    hf_output = hf_attn(x_verify)[0]
    custom_output = verify_mha(x_verify, causal=False)

max_diff = (hf_output - custom_output).abs().max().item()
print("HF output shape:", hf_output.shape)
print("Custom output shape:", custom_output.shape)
print("Maximum absolute difference:", max_diff)
print("Outputs match within tolerance:", torch.allclose(hf_output, custom_output, atol=1e-5))